In [25]:
%pip install datasets evaluate transformers accelerate peft bitsandbytes
%pip install sacrebleu

We load the dataset from Hugging Face.

In [26]:
from datasets import load_dataset

raw_datasets = load_dataset("xmj2002/Chinese_modern_classical")

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 972467
    })
})


Let's take a closer look at the features of the training set:

In [27]:
raw_datasets["train"].features

{'info': Value('string'),
 'modern': Value('string'),
 'classical': Value('string')}

Let us take a look at the translations of the first two sentences:

In [28]:
raw_datasets["train"][:2]["modern"]

['故意露出一些破绽，以引诱敌人深入我方，乘机切断他的后援和前应，最终陷他于死地。',
 '这就如《易经》 噬嗑 卦中说的，咬坚硬的腊肉而伤了牙齿一样，敌人为贪求不应得的利益，必招致后患。']

In [29]:
raw_datasets["train"][:2]["classical"]

['假之以便，唆之使前，断其援应，陷之死地。', '遇毒，位不当也。']

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Simplified Chinese -> Traditional Chinese pair.
We use `zho_Hans` for Simplified Chinese and `zho_Hant` for Traditional Chinese.

In [30]:
max_tok_length = 128 # Increased length for Chinese characters

from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
src_code = "zho_Hans"
tgt_code = "zho_Hant"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=src_code, 
    tgt_lang=tgt_code, 
    truncation=True, 
    max_length=max_tok_length,
    )

We define the preprocessing function to map `modern` to source and `classical` to target.

In [31]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["modern"], 
        text_target = sample["classical"],
        )
    return model_inputs

Check the preprocessing:

In [32]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "modern": list(sample["modern"]),
    "classical": list(sample["classical"]),
})
print(model_input)

{'input_ids': [[256200, 166675, 249485, 252223, 249191, 76163, 250944, 3, 248079, 249120, 250101, 254770, 253110, 248624, 250590, 249507, 248956, 249279, 248079, 253755, 250857, 249999, 250559, 19763, 250475, 250569, 249249, 249389, 250670, 248079, 188158, 253587, 248968, 250079, 249900, 249242, 253935, 2], [256200, 29299, 249718, 249978, 3, 250911, 250557, 3, 248059, 3, 248059, 3, 249054, 250102, 248506, 248079, 255389, 253016, 252902, 248506, 254157, 251284, 249856, 252901, 249568, 252643, 3, 88193, 248079, 253110, 248624, 249685, 254883, 249961, 249215, 250670, 249652, 248506, 37572, 248079, 249494, 252150, 251153, 250475, 250896, 253935, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[256201, 248059, 252034, 250005, 57987, 248079, 252978,

In [33]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['zho_Hans', '▁故', '意', '露', '出', '一些', '破', '<unk>', ',', '以', '引', '诱', '敌', '人', '深', '入', '我', '方', ',', '乘', '机', '切', '断', '他的', '后', '援', '和', '前', '应', ',', '最终', '陷', '他', '于', '死', '地', '。', '</s>']
['zho_Hans', '▁这', '就', '如', '<unk>', '易', '经', '<unk>', '▁', '<unk>', '▁', '<unk>', '中', '说', '的', ',', '咬', '坚', '硬', '的', '腊', '肉', '而', '伤', '了', '牙', '<unk>', '一样', ',', '敌', '人', '为', '贪', '求', '不', '应', '得', '的', '利益', ',', '必', '招', '致', '后', '患', '。', '</s>']


In [34]:
tokenizer.batch_decode(model_input['input_ids'])

['zho_Hans 故意露出一些破<unk>,以引诱敌人深入我方,乘机切断他的后援和前应,最终陷他于死地。</s>',
 'zho_Hans 这就如<unk>易经<unk> <unk> <unk>中说的,咬坚硬的腊肉而伤了牙<unk>一样,敌人为贪求不应得的利益,必招致后患。</s>']

Apply preprocessing to the dataset. Note: This dataset only has a 'train' split. We should probably split it into train/test/validation if we want to evaluate properly, but for this baseline notebook we will just use a subset for testing or split it now.

In [35]:
# Split the dataset since it only has 'train'
# We select 12000 samples: 10000 train, 1000 validation, 1000 test
shuffled_dataset = raw_datasets["train"].shuffle(seed=777).select(range(12000))

# Split into train (10000) and rest (2000)
train_testvalid = shuffled_dataset.train_test_split(test_size=2000, seed=777)

# Split rest (2000) into validation (1000) and test (1000)
test_valid = train_testvalid["test"].train_test_split(test_size=1000, seed=777)

from datasets import DatasetDict
raw_datasets = DatasetDict({
    'train': train_testvalid['train'],
    'valid': test_valid['train'],
    'test': test_valid['test']
})
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 10000
    })
    valid: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
})


In [36]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter by length:

In [37]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= max_tok_length and len(x["labels"]) <= max_tok_length , desc=f"Discarding source and target sentences with more than {max_tok_length} tokens")

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/10000 [00:00<?, ? example…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

In [38]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 4  12
 5  31
 6  48
 7  31
 8  42
 9  75
10 128
11 166
12 175
13 201
14 241
15 304
16 243
17 283
18 300
19 279
20 276
21 281
22 281
23 295
24 268
25 245
26 255
27 266
28 251
29 267
30 227
31 213
32 204
33 221
34 200
35 184
36 203
37 166
38 139
39 157
40 152
41 124
42 127
43 151
44 124
45 105
46 113
47 107
48 105
49  89
50  79
51  66
52  72
53  73
54  87
55  69
56  75
57  50
58  61
59  56
60  55
61  67
62  34
63  45
64  43
65  42
66  40
67  25
68  40
69  36
70  22
71  29
72  28
73  17
74  24
75  26
76  23
77  24
78  17
79  15
80  14
81  17
82  14
83  12
84  12
85   9
86  11
87  13
88  13
89   7
90   5
91   7
92   6
93   9
94   8
95   7
96   6
97   4
98   5
99   4
100   4
101   7
102   6
103   3
104   7
105   4
106   2
107   5
108   2
109   3
110   3
111   2
112   5
113   3
114   2
115   1
116   4
117   4
118   4
119   3
120   5
121   3
122   1
124   2
126   1
128   1


Load model with quantization:

In [39]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [40]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )

## Evaluation

In [41]:
from evaluate import load

metric = load("sacrebleu")

In [42]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Inference

In [43]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



In [44]:
test_batch_size = 32
# Use 'test' split we created
batch_tokenized_test = tokenized_datasets['test'].batch(test_batch_size)

Batching examples:   0%|          | 0/996 [00:00<?, ? examples/s]

In [45]:
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Verify target language ID
tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_code)
print(f"Target Language Code: {tgt_code}")
print(f"Target Language ID: {tgt_lang_id}")

number_of_batches = len(batch_tokenized_test["modern"])
output_sequences = []

print(f"Starting inference on {number_of_batches} batches...")
for i in tqdm(range(number_of_batches), desc="Translating"):
    inputs = tokenizer(
        batch_tokenized_test["modern"][i], 
        max_length=max_tok_length, 
        truncation=True, 
        return_tensors="pt", 
        padding=True,
        )
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=inputs["input_ids"].to(device), 
            attention_mask=inputs["attention_mask"].to(device), 
            forced_bos_token_id=tgt_lang_id, 
            max_length = max_tok_length, 
            num_beams=1, 
            do_sample=False,
            )
    output_sequences.extend(output_batch.cpu())

Using device: cuda
Target Language Code: zho_Hant
Target Language ID: 256201
Starting inference on 32 batches...


Translating:   0%|          | 0/32 [00:00<?, ?it/s]

In [46]:
result = compute_metrics((output_sequences,tokenized_datasets["test"]["labels"]))
print(f'BLEU score: {result["bleu"]}')

BLEU score: 0.3437


In [47]:
# Display some examples
num_examples = 5
print(f"--- Displaying {num_examples} random examples ---")

import random
indices = random.sample(range(len(output_sequences)), num_examples)

for i in indices:
    pred_seq = output_sequences[i]
    # Decode prediction
    pred_text = tokenizer.decode(pred_seq, skip_special_tokens=True)
    
    # Get source and reference from dataset
    item = tokenized_datasets["test"][i]
    
    # Try to get raw text if available, otherwise decode
    if "modern" in item:
        src_text = item["modern"]
    else:
        src_text = tokenizer.decode(item["input_ids"], skip_special_tokens=True)
        
    if "classical" in item:
        ref_text = item["classical"]
    else:
        # Handle -100 in labels
        label_ids = [l if l != -100 else tokenizer.pad_token_id for l in item["labels"]]
        ref_text = tokenizer.decode(label_ids, skip_special_tokens=True)
    
    print(f"Example {i}:")
    print(f"Source (Modern):       {src_text}")
    print(f"Reference (Classical): {ref_text}")
    print(f"Prediction:            {pred_text}")
    print("-" * 80)

--- Displaying 5 random examples ---
Example 368:
Source (Modern):       封德彝讲的话深得大体，我心悦诚服，不敢有所非议。 
Reference (Classical): 德彝所言，真得大体，臣诚心服，不敢遂非。 
Prediction:            我對這話有深厚的信心,我毫無無猶.
--------------------------------------------------------------------------------
Example 292:
Source (Modern):       正元、景元初年，朝廷连续两次给他增加食邑，共四千四百户。
Reference (Classical): 正元、景元初，连增邑，凡四千四百户。
Prediction:            西元初年,朝廷連續兩次給他加了四千四百個.
--------------------------------------------------------------------------------
Example 553:
Source (Modern):       氐人豪族仇檀又起兵反叛，陆真前去讨平，修建完长蛇镇后返回。
Reference (Classical): 氐豪仇傉檀反，真讨平之，卒城而还。
Prediction:            族的族敵人又起了反抗,
--------------------------------------------------------------------------------
Example 145:
Source (Modern):       涓水又往东流经陆浑县老城北面。
Reference (Classical): 涓水又东径陆浑县故城北。
Prediction:            河河又向東流,穿越縣老城北部.
--------------------------------------------------------------------------------
Example 206:
Source (Modern):       九月五日，中书令萧嵩等人奉上《开元新礼》一百五十卷，诏命所管部门施行